In [ ]:
!pip install -q transformers==4.46.3 datasets accelerate evaluate rouge-score bert-score sacrebleu sentencepiece peft bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 56.1 MB/s eta 0:00:00


In [ ]:
import os
import gc
import shutil
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sacrebleu

from tqdm import tqdm
from datasets import Dataset
from rouge_score import rouge_scorer
from bert_score import score as bertscore

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
BASE_DIR = "/content/drive/MyDrive/SARVAM_Malayalam_Summarization"

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(f"{BASE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{BASE_DIR}/results", exist_ok=True)

print(BASE_DIR)

/content/drive/MyDrive/SARVAM_Malayalam_Summarization


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving validation.csv to validation.csv
Saving train.csv to train.csv
Saving test.csv to test.csv


In [ ]:
train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/validation.csv")
test_df = pd.read_csv("/content/test.csv")

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(7758, 3)
(970, 3)
(970, 3)


In [ ]:
for df in [train_df, val_df, test_df]:
    df.dropna(subset=["article", "summary"], inplace=True)
    df["article"] = df["article"].astype(str)
    df["summary"] = df["summary"].astype(str)

In [ ]:
def format_prompt(article, summary=None):

    prompt = f"""Summarize this Malayalam news article in Malayalam.

Article:
{article}

Summary:
"""

    if summary:
        prompt += summary

    return prompt

In [ ]:
train_df["text"] = train_df.apply(
    lambda x: format_prompt(x["article"], x["summary"]),
    axis=1
)

val_df["text"] = val_df.apply(
    lambda x: format_prompt(x["article"], x["summary"]),
    axis=1
)

In [ ]:
train_dataset = Dataset.from_pandas(train_df[["text"]])
val_dataset = Dataset.from_pandas(val_df[["text"]])

In [ ]:
MODEL_NAME = "sarvamai/sarvam-2b-v0.5"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.70M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.76G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Loaded


In [ ]:
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 1,605,632 || all params: 2,510,440,448 || trainable%: 0.0640


In [ ]:
MAX_LENGTH = 128

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

In [ ]:
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/7758 [00:00<?, ? examples/s]

Map:   0%|          | 0/970 [00:00<?, ? examples/s]

In [ ]:
 data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
training_args = TrainingArguments(
    output_dir=f"{BASE_DIR}/checkpoints",

    learning_rate=2e-4,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=16,

    num_train_epochs=5,

    eval_strategy="epoch",
    save_strategy="epoch",

    fp16=False,

    logging_steps=50,

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",

    optim="paged_adamw_8bit"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
0,1.545700,1.508905
1,1.503100,1.483479
2,1.421500,1.472159
3,1.448000,1.466666


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Epoch,Training Loss,Validation Loss
0,1.545700,1.508905
1,1.503100,1.483479
2,1.421500,1.472159
3,1.448000,1.466666
4,1.413500,1.465724


TrainOutput(global_step=2420, training_loss=1.4815583717724508, metrics={'train_runtime': 15415.4279, 'train_samples_per_second': 2.516, 'train_steps_per_second': 0.157, 'total_flos': 7.074738899779584e+16, 'train_loss': 1.4815583717724508, 'epoch': 4.9909770559422535})

In [ ]:
MODEL_SAVE = f"{BASE_DIR}/final_sarvam_model"

trainer.save_model(MODEL_SAVE)
tokenizer.save_pretrained(MODEL_SAVE)

print("Saved")

Saved


In [ ]:
shutil.make_archive(
    f"{BASE_DIR}/final_sarvam_model",
    "zip",
    MODEL_SAVE
)

'/content/drive/MyDrive/SARVAM_Malayalam_Summarization/final_sarvam_model.zip'

In [ ]:
predictions = []
references = []
articles = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):

    article = row["article"]
    reference = row["summary"]

    prompt = format_prompt(article)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=2,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    pred = generated.split("Summary:")[-1].strip()

    predictions.append(pred)
    references.append(reference)
    articles.append(article)

  0%|          | 0/970 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
 42%|████▏     | 405/970 [49:48<1:09:00,  7.33s/it]

In [ ]:
pred_df = pd.DataFrame({
    "Article": articles,
    "Reference": references,
    "Prediction": predictions
})

pred_df.to_csv(
    f"{BASE_DIR}/results/sarvam_predictions.csv",
    index=False
)

In [ ]:
for i in range(10):

    print("=" * 120)
    print(f"SAMPLE {i+1}")

    print("\nARTICLE:\n")
    print(articles[i])

    print("\nREFERENCE:\n")
    print(references[i])

    print("\nPREDICTION:\n")
    print(predictions[i])

In [ ]:
sample_df = pred_df.sample(10, random_state=42)

sample_df.to_csv(
    f"{BASE_DIR}/results/sarvam_sample10.csv",
    index=False
)

In [ ]:
rouge = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=False
)

r1 = []
r2 = []
rl = []

for ref, pred in zip(references, predictions):
    scores = rouge.score(ref, pred)

    r1.append(scores["rouge1"].fmeasure)
    r2.append(scores["rouge2"].fmeasure)
    rl.append(scores["rougeL"].fmeasure)

In [ ]:
bleu = sacrebleu.corpus_bleu(
    predictions,
    [references]
)

In [ ]:
P, R, F1 = bertscore(
    predictions,
    references,
    lang="ml"
)

In [ ]:
metrics_df = pd.DataFrame({
    "Metric": [
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "BLEU",
        "BERTScore-F1"
    ],
    "Score": [
        np.mean(r1),
        np.mean(r2),
        np.mean(rl),
        bleu.score,
        F1.mean().item()
    ]
})

metrics_df.to_csv(
    f"{BASE_DIR}/results/sarvam_metrics.csv",
    index=False
)

metrics_df

In [ ]:
logs = trainer.state.log_history
log_df = pd.DataFrame(logs)

plt.figure(figsize=(8,6))

if "loss" in log_df.columns:
    plt.plot(log_df["loss"].dropna(), label="Training Loss")

if "eval_loss" in log_df.columns:
    plt.plot(log_df["eval_loss"].dropna(), label="Validation Loss")

plt.legend()
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("SARVAM Training Curve")

plt.savefig(
    f"{BASE_DIR}/results/sarvam_training_curve.png"
)

plt.show()

In [ ]:
shutil.make_archive(
    f"{BASE_DIR}/sarvam_results",
    "zip",
    f"{BASE_DIR}/results"
)

In [ ]:
from google.colab import files

files.download(f"{BASE_DIR}/final_sarvam_model.zip")
files.download(f"{BASE_DIR}/sarvam_results.zip")